## Part 1 of this Work
We use the Penmanshiel dataset to create a controlled environment.


In this part we calibrate the threshold. This threshold will be used for part 2.

In this part we also explore the anomaly detection capabilities of the autoencoder on selected anomalies.

If further content exist for this part, you can check project_root/src/further_content/





In [ ]:
from helperfunctions import helper as hfn
from helperfunctions import intern_constants as ic
from helperfunctions.pretty_print import PrettyPrint as pp
from helperfunctions import training_lib as tl
# from helperfunctions.controlled_env import Inject_Anomalies as ia
# from helperfunctions.controlled_env import Anom_Type as anomtype
# from helperfunctions.controlled_env import Eval_Anom as ea
from torch import nn
from helperfunctions import controlled_env as ce
from helperfunctions.helper import load_feature_order
from typing import cast, Tuple
import pandas as pd
from glob import glob
import os

In [ ]:
os.makedirs(ic.PATH_PART1_VAL_SET_INJ_DIR , exist_ok=True)
os.makedirs(ic.PATH_PART1_VAL_LOSS_DIR, exist_ok=True)
os.makedirs(ic.PATH_PART1_TEST_LOSS_DIR, exist_ok=True)
os.makedirs(ic.PATH_PART1_K_AGG_METRICS_DIR, exist_ok=True)

out_dir = ic.PATH_PART1_K_AGG_METRICS_DIR

fp_per_wt_k = out_dir / "per_wt_k_metrics.csv"
fp_per_event =  out_dir / "per_event_metrics.csv"
fp_best_k = out_dir / "best_ks_per_wt.csv"
fp_anom_view =  out_dir / "anom_overview.csv"
fp_thresh =  out_dir / "thresholds_part2.csv"
fp_t_grid =  out_dir / "threshold_grid.csv"
fp_sigma = out_dir / "sigma_val_mean.csv"
fp_cand = out_dir / "candidates_k_per_wt.csv"

### Data set creation

In [ ]:
cfg_dummy = hfn.TrainConfig(config_name="p1_dummy_cfg", part1=True)

In [ ]:
print(cfg_dummy.val_start_time)
print(cfg_dummy.val_end_time)

Anomaly injection time period

In [ ]:
print(ic.START_ANOM)
print(ic.END_ANOM)

In [ ]:
fo = hfn.load_feature_order()
print(*fo, sep="\n")

In [ ]:
wt = 1

SIG_ADD =  "Rotor bearing temp (°C)"
SIG_POINT= "Stator temperature 1 (°C)"
SIG_MULT = "Transformer cell temperature (°C)"
SIG_CORR1= "Generator bearing front temperature (°C)"
SIG_CORR2= "Generator bearing rear temperature (°C)"

intensities_add = [15, 25.0]
intensities_mult = [1.2]
intensities_corr = [-0.9, 0.0, 0.8]
intensities_point = [50, 100]

specs: list[ce.AnomalySpec] = [
    ce.AnomalySpec(
        wt_id= wt,
        category=ce.AnomCategory.ADD,
        primary_signal=SIG_ADD,
        intensities=intensities_add,
    ),
    ce.AnomalySpec(
        wt_id= wt,
        category=ce.AnomCategory.MULT,
        primary_signal=SIG_MULT,
        intensities=intensities_mult,
    ),
    ce.AnomalySpec(
        wt_id= wt,
        category=ce.AnomCategory.CORR,
        primary_signal=SIG_CORR1,
        secondary_signal=SIG_CORR2,
        intensities=intensities_corr,
    ),
    ce.AnomalySpec(
        wt_id= wt,
        category=ce.AnomCategory.POINT,
        primary_signal=SIG_POINT,
        intensities=intensities_point,
    ),
]

print(specs)

In [ ]:
n_windows_by_cat = ce.Part1.infer_windows_by_category_from_specs(specs)

print(n_windows_by_cat)

In [ ]:
anomaly_plan, cat_windows  = cast(Tuple[ce.AnomalyPlan, pd.DataFrame] ,ce.Part1.build_plan( 
    start=ic.START_ANOM,
    end=ic.END_ANOM,
    gap_in_hours=48,
    min_len_hours=168,
    k_add=n_windows_by_cat[ce.AnomCategory.ADD],
    k_point=n_windows_by_cat[ce.AnomCategory.POINT],
    k_mult=n_windows_by_cat[ce.AnomCategory.MULT],
    k_corr=n_windows_by_cat[ce.AnomCategory.CORR],
    return_cat_windows=True))

cat_win_path = out_dir / "anom_windows.csv"
cat_windows.to_csv(cat_win_path, index=False, date_format="%Y-%m-%d %H:%M:%S")

display(anomaly_plan)
display(cat_windows)

In [ ]:
dfs = []
# 
files = glob(os.path.join(ic.PATH_IMPUTED, "*.csv"))
for filepath in files:
    df = pd.read_csv(filepath, parse_dates=[ic.TS_COL])
    dfs.append(df)
    
df_val2_part1_raw = pd.concat(dfs, ignore_index=True)
df_val2_part1_raw[ic.TS_COL] = pd.to_datetime(df_val2_part1_raw[ic.TS_COL], errors="raise")

In [ ]:
df_val_injected, df_gt = ce.Part1.apply_plan_and_build(df_raw_sigs=df_val2_part1_raw,
                                                        specs=specs,
                                                        plan= anomaly_plan)

gt_union_df = ce.Part1._gt_union_per_wt(df_gt)

display(gt_union_df.head())

df_gt.to_csv(out_dir / "gt_events_part1", index=False)

In [ ]:
for wt_id in sorted(df_val_injected[ic.WT_ID].unique()):
    df_wt_test = df_val_injected[df_val_injected[ic.WT_ID] == wt_id].copy()
    fp = ic.PATH_PART1_VAL_SET_INJ_DIR / f"wt_{int(wt_id)}_part1_val_set_injected.csv"
    df_wt_test.to_csv(fp, index=False)

In [ ]:
_, val_loader, val_inj_loader = hfn.build_dataloaders(
    train_csv_dir=ic.PATH_PC_FILTERING,
    val_csv_dir=ic.PATH_IMPUTED,
    test_csv_dir=ic.PATH_PART1_VAL_SET_INJ_DIR,
    cfg=cfg_dummy
)

In [ ]:
best = tl.get_model_results(ic.PATH_TO_BEST_MODEL_DIR, best_n=1)
ae, optim, ckpt, sched, scaler = best[0]

ae = ae.to(cfg_dummy.device).eval()

In [ ]:
loss_fn = nn.MSELoss(reduction="none")

In [ ]:
val_df_eval = tl.eval_model(
    model=ae,
    data_loader=val_loader,
    device=cfg_dummy.device,
    loss_fn = loss_fn
)

val_df_eval.to_csv(ic.PATH_PART1_VAL_LOSS_DIR/ "df_val_eval.csv", index=False)

prep_val_df_eval = ce.Part1.prepare_df_eval_threshold_computing(df_eval=val_df_eval,
                                                                include_mean=True,
                                                                include_signal_wise_re=False,
                                                                )

display(prep_val_df_eval.head())





In [ ]:
sigma_df = ce.Part1.compute_val_sigma(re_val_df=prep_val_df_eval)
sigma_df.to_csv(fp_sigma, index=False)

In [ ]:
try:
    val_inj_df_eval = pd.read_csv(ic.PATH_PART1_TEST_LOSS_DIR/ "df_inj_eval.csv")
except FileNotFoundError:
    
    val_inj_df_eval  =tl.eval_model(
        model=ae,
        data_loader=val_inj_loader,
        device=cfg_dummy.device,
        loss_fn= loss_fn
    )

    val_inj_df_eval.to_csv(ic.PATH_PART1_TEST_LOSS_DIR/ "df_inj_eval.csv", index=False)

prep_val_inj_df_eval = ce.Part1.prepare_df_eval_threshold_computing(df_eval=val_inj_df_eval,
                                                                 include_mean=True,
                                                                 include_signal_wise_re=False)


In [ ]:

# k_values = ce.Part1._k_grid(max=10, min=1, step=1)
# k_values2 = ce.Part1._k_grid(max=1,min=0.1, step=0.1)



# k_values = (k_values + 
#             k_values2 + 
#             )

k_values = ce.Part1._k_grid(max=1, min=0.9, step=0.1)
k2_values = ce.Part1._k_grid(max=3.1, min=2.9, step=0.1)

k_values = k_values + k2_values

In [ ]:
thresholds_grid_df = ce.Part1.build_threshold_grid(
    sigma_df=sigma_df,
    k_values= k_values,
)

display(thresholds_grid_df.head())

thresholds_grid_df.to_csv(fp_t_grid, index=False)

In [ ]:
per_wt_k_df, per_event_df = ce.Part1.eval_events_over_k(
    wt_sig_loss_df=prep_val_inj_df_eval,
    gt_union_df=gt_union_df,
    gt_events_df=df_gt,
    thresholds_grid_df=thresholds_grid_df,
    fp_merge_gap_steps=0,
)

per_wt_k_df.to_csv(fp_per_wt_k, index=False)
per_event_df.to_csv(fp_per_event, index=False)

In [ ]:
display(per_wt_k_df.head(20))
display(per_event_df.head(10))

In [ ]:
best_k_per_wt_df = ce.Part1.select_k_per_wt(
    per_wt_k_df= per_wt_k_df,
    save_dir=out_dir,
    csv_name=fp_cand.name,
)

best_k_per_wt_df.to_csv(fp_best_k, index=False)

In [ ]:
display(best_k_per_wt_df)

In [ ]:
anom_overview_df = ce.Part1.build_target_table(
    per_event_df = per_event_df,
    best_k_per_wt=best_k_per_wt_df,
    save_dir=out_dir,
    csv_name=fp_anom_view.name,
)

display(anom_overview_df.head(10))

In [ ]:
cat_win_path = ic.PATH_PART1_K_AGG_METRICS_DIR / "anom_windows.csv"

cat_windows = pd.read_csv(cat_win_path)

spans_all = list(zip(pd.to_datetime(cat_windows[ce.AnomOverviewKeys.TS_START]),
                     pd.to_datetime(cat_windows[ce.AnomOverviewKeys.TS_END])))

for i in spans_all:
    print(i)

fp_best_k = ic.PATH_PART1_K_AGG_METRICS_DIR / "best_ks_per_wt.csv"

threshold_df = pd.read_csv(fp_best_k)
display(threshold_df.head(10))
theta = threshold_df["threshold"].iloc[0]

pp.print_loss(val_inj_df_eval,
              dpi=300,
              y_limits=((0,0.1),(0.1,0.7)),
              title="Test Set: Mean Turbine Loss per Sample",
              wt_id= [1],
            #   ts_range=("2018-08-01 00:00:00","2018-10-28 00:00:00"),
              ts_range=(ic.START_ANOM, '2020-03-12 00:00:00'),
              anomaly_spans=spans_all,
              line_width=0.8,
              mark_threshold=theta,
              show_mean=False)